In [ ]:
import pandas as pd
import numpy as np
import pip
! pip install matplotlib
! pip install seaborn
! pip install scipy
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro

In [ ]:
df = pd.read_csv('quotes_all.csv')
df

In [ ]:
df = df.drop(columns=['Unnamed: 0'])
df['Date'] = pd.to_datetime(df['Date'])
df['Volume'] = df['Volume'].astype(int)
df.info()

In [ ]:
df.drop_duplicates
df.shape[0]

In [ ]:
df.isna().sum()

В датафрейме нет пропусков и дубликатов, он готов к работе.

Добавим delta - столбец отношения close date текущего и предыдущего дня. Для первого дня заполним потенциальный пропуск нулем.

In [ ]:
df['delta'] = df.groupby('ticker')['Close'].transform( lambda x: x / x.shift(1)).fillna(0)

Логарифмируем его, возможно это пригодится далее для анализа:

In [ ]:
df['ln_delta'] = np.log(df['delta'].replace(0, np.nan)).fillna(0)

In [ ]:
df

In [ ]:
df['ticker'].value_counts()

In [ ]:
df_aapl  = df[df['ticker'] == 'AAPL']
df_xom  = df[df['ticker'] == 'XOM']
df_tsla = df[df['ticker'] == 'TSLA']
df_wmt = df[df['ticker'] == 'WMT']
df_pfe = df[df['ticker'] == 'PFE']
df_nflx = df[df['ticker'] == 'NFLX']
df_jpm = df[df['ticker'] == 'JPM']
df_cat = df[df['ticker'] == 'CAT']

ticker_dfs = [df_aapl, df_xom, df_tsla, df_wmt, df_pfe, df_nflx, df_jpm, df_cat]

In [ ]:
for dft in ticker_dfs:
    print("\n")
    print(f"Компания: {dft['ticker'].iloc[0]}")
    print(dft.describe())

In [ ]:
for dft in ticker_dfs:
    data = dft['Date']

In [ ]:
df['Year'] = df['Date'].dt.year

pivot_table = pd.crosstab(df['ticker'], df['Year'])

pivot_table

Число наблюдений совпадает для всех компаний, что подтверждает отсутствие ошибок при сборе данных.

In [ ]:
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'delta', 'ln_delta']

Посмотрим на распределение значений во всех числовых колонках для акций всех компаний по-отдельности.

In [ ]:
for dft in ticker_dfs:
    print(f"Компания: {dft['ticker'].iloc[0]}")
    for col in numeric_cols:
        fig, axes = plt.subplots(1, 3, figsize=(20, 4))
        fig.suptitle(f'{col}', fontsize=14)
        sns.histplot(dft[col], ax=axes[0])
        axes[0].set_title('Гистограмма')
        sns.boxplot(x=dft[col], ax=axes[1])
        axes[1].set_title('Ящик с усами')
        sns.violinplot(x=dft[col], ax=axes[2])
        axes[2].set_title('Скрипичная диаграмма')
        plt.tight_layout()
        plt.show()

Заметим что:
1) Open, High, Low, Close, Adj Close в рамках одной компании практически идентичны
2) У распределения цен имеется 2-3 пика
3) Распределение delta визуально близко к нормальному, проверим, так ли это:

In [ ]:
for dft in ticker_dfs:
    print("\n")
    print(f"Компания: {dft['ticker'].iloc[0]}")

    stat_delta, p_delta = shapiro(dft['delta'].values)
    print(f"delta: значение статистики = {stat_delta:.4f}, p-value={p_delta:.4f}")
    if p_delta > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")

    stat_ln, p_ln = shapiro(dft['ln_delta'].values)
    print(f"ln_delta: значение статистики = {stat_ln:.4f}, p-value={p_ln:.4f}")
    if p_ln > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")

Убедимся, что на более маленьких случайных подвыборках результаты будут аналогичными:

In [ ]:
for dft in ticker_dfs:
    print("\n")
    print(f"Компания: {dft['ticker'].iloc[0]}")

    stat_delta, p_delta = shapiro(dft['delta'].sample(100, random_state=69).values)
    print(f"delta: значение статистики = {stat_delta:.4f}, p-value={p_delta:.4f}")
    if p_delta > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")

    stat_ln, p_ln = shapiro(dft['ln_delta'].sample(100, random_state=69).values)
    print(f"ln_delta: значение статистики = {stat_ln:.4f}, p-value={p_ln:.4f}")
    if p_ln > 0.05:
        print("Распределение статистически не отличается от нормального (p > 0.05)")
    else:
        print("Распределение не является нормальным (p <= 0.05)")


Таким образом, это все же не нормальное распределение.

Те же визуализации для общего датасета по всем компаниям:

In [ ]:
for col in numeric_cols:
    fig, axes = plt.subplots(1, 3, figsize=(20, 4))
    fig.suptitle(f'{col}', fontsize=14)
    sns.histplot(df[col], ax=axes[0])
    axes[0].set_title('Гистограмма')
    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title('Ящик с усами')
    sns.violinplot(x=df[col], ax=axes[2])
    axes[2].set_title('Скрипичная диаграмма')
    plt.tight_layout()
    plt.show()

Графики демонстируют, что в общем датасете содержится смесь распределений для акций разных компаний.

В общих данных и в данных по компаниям поотдельности есть много выбросов, однако удалять их нет смысла, так как они представляют интерес в рамках поставленной задачи - модель должна прелсказывать в том числе и резкие колебания цен, на которых в первую очередь можно заработать.